# Homework 5 — Texture Mapping and Multiple Static Lights

In [2]:
import os
import sys
import ctypes
import numpy as np
from PIL import Image
import glfw
from OpenGL.GL import *

TEXTURE_FILE = "goku.jpg"

In [3]:
# ==============================================================================
# 1. GLSL Shader Source Code
# ==============================================================================

VERTEX_SHADER_SRC = """#version 330 core
layout(location = 0) in vec3 aPos;  // Vertex 3D position
layout(location = 1) in vec2 aUv;   // 2D UV texture coordinate

uniform mat4 uModelView;
uniform mat4 uPersp;

out vec3 vPoint; // Interpolated camera/eye-space position
out vec2 vUv;    // Interpolated UV coordinates passed to fragment shader

void main() {
    // Transform vertex position into camera space
    vPoint = (uModelView * vec4(aPos, 1.0)).xyz;

    // Project camera-space coordinates into 4D clip space
    gl_Position = uPersp * vec4(vPoint, 1.0);

    // Pass UV coordinates directly down to rasterizer
    vUv = aUv;
}
"""

FRAGMENT_SHADER_SRC = """#version 330 core
in vec3 vPoint;
in vec2 vUv;

out vec4 FragColor;

uniform sampler2D uTexture; // 2D texture sampler linked to texture unit 0

// Multiple static lights passed in camera/eye space
uniform int uNumLights;
uniform vec3 uLights[8];

// Illumination coefficient weights
uniform float uAmb = 0.20;  // Ambient reflection weight
uniform float uDif = 0.70;  // Diffuse reflection weight
uniform float uSpc = 0.50;  // Specular reflection weight

void main() {
    // 1. Analytical face normal using screen-space partial derivatives (faceted appearance)
    vec3 dx = dFdx(vPoint);
    vec3 dy = dFdy(vPoint);
    vec3 N = normalize(cross(dx, dy));

    // Eye vector: camera sits at origin (0, 0, 0) in camera space
    vec3 E = normalize(-vPoint);

    // 2. Accumulate lighting contributions across all static lights
    float totalDiffuse = 0.0;
    float totalSpecular = 0.0;

    for (int i = 0; i < uNumLights; i++) {
        // Light vector pointing from surface to light source
        vec3 L = normalize(uLights[i] - vPoint);

        // Lambertian diffuse component
        float diff = max(0.0, dot(N, L));
        totalDiffuse += diff;

        // Phong specular highlight
        vec3 R = reflect(-L, N);
        float specAngle = max(0.0, dot(R, E));
        if (diff > 0.0) totalSpecular += pow(specAngle, 32.0);
    }
    // 3. multiplies lighting scalar across the RGB vector of the texture
    vec3 texColor = texture(uTexture, vUv).rgb;
    // 4. Combine ambient base with accumulated diffuse and specular terms
    float baseLight = min(1.0, uAmb + uDif * totalDiffuse);
    vec3 finalColor = baseLight * texColor + vec3(uSpc * totalSpecular);
    FragColor = vec4(clamp(finalColor, 0.0, 1.0), 1.0);
}
"""

In [4]:
# ==============================================================================
# 2. Geometry Builder with Automated UV Mapping
# ==============================================================================

NUM_FRONT = 10
Z_FRONT, Z_BACK = 0.10, -0.10


def compute_planar_uvs(points_2d):
    min_xy = points_2d.min(axis=0)
    max_xy = points_2d.max(axis=0)
    return ((points_2d - min_xy) / (max_xy - min_xy)).astype(np.float32)


def build_textured_letter_n():
    """
    Builds the 3D extruded letter N, mapping UV coordinates to [0, 1]^2
    using the bounding box of the front face.
    """
    front_2d = np.array([
        [  0.0,   0.0],  # 0 A
        [ 60.0,   0.0],  # 1 B
        [ 60.0, 210.0],  # 2 C
        [240.0,   0.0],  # 3 D
        [300.0,   0.0],  # 4 E
        [300.0, 300.0],  # 5 F
        [240.0, 300.0],  # 6 G
        [240.0,  90.0],  # 7 H
        [ 60.0, 300.0],  # 8 I
        [  0.0, 300.0],  # 9 J
    ], dtype=np.float32)

    # 1. Generate UVs mapped to [0, 1] based on bounds before scaling
    uvs_front = compute_planar_uvs(front_2d)
    # Both front and back faces share identical UV projections
    uvs = np.vstack([uvs_front, uvs_front])

    # 2. Scale 3D coordinates to fit within the requested bounds
    center_xy = (front_2d.max(axis=0) + front_2d.min(axis=0)) / 2.0
    front_2d = (front_2d - center_xy) * (1.6 / 300.0)

    front_verts = np.column_stack([front_2d, np.full(NUM_FRONT, Z_FRONT, dtype=np.float32)])
    back_verts = np.column_stack([front_2d, np.full(NUM_FRONT, Z_BACK, dtype=np.float32)])
    vertices = np.vstack([front_verts, back_verts]).astype(np.float32)

    # 3. Build Triangles
    front_tris = [
        [0, 1, 2], [0, 2, 9], [2, 8, 9],
        [2, 3, 7], [2, 7, 8],
        [3, 4, 7], [7, 4, 5], [7, 5, 6],
    ]

    # Back fan: reverse winding order so normal points outward (-Z)
    back_tris = [[a + NUM_FRONT, c + NUM_FRONT, b + NUM_FRONT] for a, b, c in front_tris]

    # Side walls: quads formed along the outer loop connecting front to back
    side_tris = []
    for i in range(NUM_FRONT):
        j = (i + 1) % NUM_FRONT
        i_back, j_back = i + NUM_FRONT, j + NUM_FRONT
        side_tris.append([i, i_back, j])
        side_tris.append([j, i_back, j_back])

    indices = np.array(front_tris + back_tris + side_tris, dtype=np.uint32).flatten()
    return vertices, uvs, indices

In [5]:
_v, _uv, _idx = build_textured_letter_n()
assert _uv.shape == (20, 2) and _uv.min() >= 0.0 and _uv.max() <= 1.0
assert np.allclose(_uv.min(axis=0), 0.0) and np.allclose(_uv.max(axis=0), 1.0)
assert np.array_equal(_uv[:NUM_FRONT], _uv[NUM_FRONT:])
print(f"{len(_v)} vertices, {len(_idx) // 3} triangles; UV range {_uv.min(axis=0)} .. {_uv.max(axis=0)}")
print(np.column_stack([_v[:NUM_FRONT, :2], _uv[:NUM_FRONT]]).round(3))

20 vertices, 36 triangles; UV range [0. 0.] .. [1. 1.]
[[-0.8  -0.8   0.    0.  ]
 [-0.48 -0.8   0.2   0.  ]
 [-0.48  0.32  0.2   0.7 ]
 [ 0.48 -0.8   0.8   0.  ]
 [ 0.8  -0.8   1.    0.  ]
 [ 0.8   0.8   1.    1.  ]
 [ 0.48  0.8   0.8   1.  ]
 [ 0.48 -0.32  0.8   0.3 ]
 [-0.48  0.8   0.2   1.  ]
 [-0.8   0.8   0.    1.  ]]


In [6]:
# ==============================================================================
# 3. Texture Loading & GPU Allocation
# ==============================================================================

def make_checkerboard(size=256, cell=32):
    grid = (np.indices((size, size)) // cell).sum(axis=0) % 2
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[grid == 0] = [220, 70, 50]
    img[grid == 1] = [235, 235, 240]
    return img


def load_texture(filepath):
    try:
        # Open and strictly force 3-channel RGB mode
        img = Image.open(filepath).convert("RGB").transpose(Image.Transpose.FLIP_TOP_BOTTOM)
        # Ensure memory is contiguous C-order bytes
        img_data = np.ascontiguousarray(np.asarray(img, dtype=np.uint8))
        print(f"Loaded RGB texture successfully from: {filepath} ({img.width}x{img.height})")
    except (FileNotFoundError, OSError) as e:
        print(f"Warning: could not load '{filepath}' ({e}). Generating checkerboard fallback.")
        img_data = make_checkerboard()

    h, w = img_data.shape[:2]

    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)

    # NOTE: Tell OpenGL byte rows are tightly packed with 1-byte alignment (RGB = 3 bytes/pixel)
    glPixelStorei(GL_UNPACK_ALIGNMENT, 1)

    # Filtering & wrap modes
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    # Upload pixels as GL_RGB
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, w, h, 0, GL_RGB, GL_UNSIGNED_BYTE, img_data)
    glGenerateMipmap(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, 0)
    return tex_id

In [7]:
# ==============================================================================
# 4. Camera & Matrix Math
# ==============================================================================

class ArcballCamera:
    """ Mouse-driven orbit camera operating on pitch/yaw angles."""
    def __init__(self, distance=3.2):
        self.distance = distance
        self.rot_x = 20.0   # Pitch
        self.rot_y = -35.0  # Yaw
        self.last_x = 0.0
        self.last_y = 0.0
        self.is_dragging = False

    def on_press(self, x, y):
        self.is_dragging = True
        self.last_x, self.last_y = x, y

    def on_release(self):
        self.is_dragging = False

    def on_drag(self, x, y):
        if not self.is_dragging:
            return
        dx, dy = x - self.last_x, y - self.last_y
        self.last_x, self.last_y = x, y
        self.rot_y += dx * 0.4
        self.rot_x = float(np.clip(self.rot_x + dy * 0.4, -89.0, 89.0))

    def get_view_matrix(self):
        ax, ay = np.radians(self.rot_x), np.radians(self.rot_y)
        Rx = np.array([[1, 0, 0, 0],
                       [0, np.cos(ax), -np.sin(ax), 0],
                       [0, np.sin(ax),  np.cos(ax), 0],
                       [0, 0, 0, 1]], dtype=np.float32)
        Ry = np.array([[ np.cos(ay), 0, np.sin(ay), 0],
                       [0, 1, 0, 0],
                       [-np.sin(ay), 0, np.cos(ay), 0],
                       [0, 0, 0, 1]], dtype=np.float32)
        T = np.identity(4, dtype=np.float32)
        T[2, 3] = -self.distance
        return T @ Rx @ Ry


def make_perspective(fov_rad, aspect, near, far):
    tan_half = np.tan(fov_rad / 2.0)
    P = np.zeros((4, 4), dtype=np.float32)
    P[0, 0] = 1.0 / (aspect * tan_half)
    P[1, 1] = 1.0 / tan_half
    P[2, 2] = -(far + near) / (far - near)
    P[2, 3] = -(2.0 * far * near) / (far - near)
    P[3, 2] = -1.0
    return P


def lights_to_eye_space(view, world_lights):
    homog = np.hstack([world_lights, np.ones((len(world_lights), 1), dtype=np.float32)])
    return np.ascontiguousarray((view @ homog.T).T[:, :3], dtype=np.float32)


# 3 static lights balanced around the model: Front-Right, Front-Left, and Back
WORLD_LIGHTS = np.array([
    [ 2.5,  2.0,  3.0],  # Light 1: Front-top-right key light
    [-2.5, -1.0,  2.0],  # Light 2: Front-bottom-left fill light
    [ 0.0,  1.5, -3.5],  # Light 3: Back light illuminating the rear face & side walls
], dtype=np.float32)
assert len(WORLD_LIGHTS) <= 8

In [8]:
_cam = ArcballCamera()
_cam.rot_x = _cam.rot_y = 0.0
assert np.allclose(lights_to_eye_space(_cam.get_view_matrix(), WORLD_LIGHTS),
                   WORLD_LIGHTS - [0, 0, _cam.distance])
print(lights_to_eye_space(ArcballCamera().get_view_matrix(), WORLD_LIGHTS).round(3))

[[ 0.327  0.548  1.141]
 [-3.195 -1.01  -3.35 ]
 [ 2.008  2.39  -5.381]]


In [9]:
# ==============================================================================
# 5. Shader Compilation Helper
# ==============================================================================

def compile_shader(src, stype):
    s = glCreateShader(stype)
    glShaderSource(s, src)
    glCompileShader(s)
    if not glGetShaderiv(s, GL_COMPILE_STATUS):
        raise RuntimeError(glGetShaderInfoLog(s).decode())
    return s


def build_program():
    vs = compile_shader(VERTEX_SHADER_SRC, GL_VERTEX_SHADER)
    fs = compile_shader(FRAGMENT_SHADER_SRC, GL_FRAGMENT_SHADER)
    p = glCreateProgram()
    glAttachShader(p, vs)
    glAttachShader(p, fs)
    glLinkProgram(p)
    if not glGetProgramiv(p, GL_LINK_STATUS):
        raise RuntimeError(glGetProgramInfoLog(p).decode())
    glDeleteShader(vs)
    glDeleteShader(fs)
    return p

In [10]:
# ==============================================================================
# 6. Main Execution Loop
# ==============================================================================

def main():
    if not glfw.init():
        sys.exit("Failed to initialize GLFW.")
    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)
    glfw.window_hint(glfw.OPENGL_FORWARD_COMPAT, GL_TRUE)

    window = glfw.create_window(800, 700, "HW5: Textured 3D Letter with Multiple Lights", None, None)
    if not window:
        glfw.terminate()
        sys.exit("Failed to create GLFW window.")
    glfw.make_context_current(window)
    glEnable(GL_DEPTH_TEST)

    camera = ArcballCamera()

    # Mouse input handlers
    def mouse_button_cb(w, btn, act, mods):
        if btn == glfw.MOUSE_BUTTON_LEFT:
            if act == glfw.PRESS:
                camera.on_press(*glfw.get_cursor_pos(w))
            elif act == glfw.RELEASE:
                camera.on_release()

    def cursor_pos_cb(w, x, y):
        camera.on_drag(x, y)

    glfw.set_mouse_button_callback(window, mouse_button_cb)
    glfw.set_cursor_pos_callback(window, cursor_pos_cb)

    # 1. Build and interleave geometry attributes: [x, y, z, u, v]
    verts, uvs, indices = build_textured_letter_n()
    interleaved = np.ascontiguousarray(np.hstack([verts, uvs]), dtype=np.float32)

    # 2. Buffers setup (VAO, VBO, EBO)
    vao = glGenVertexArrays(1)
    glBindVertexArray(vao)
    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, interleaved.nbytes, interleaved, GL_STATIC_DRAW)
    ebo = glGenBuffers(1)
    glBindBuffer(GL_ELEMENT_ARRAY_BUFFER, ebo)
    glBufferData(GL_ELEMENT_ARRAY_BUFFER, indices.nbytes, indices, GL_STATIC_DRAW)

    stride = 5 * interleaved.itemsize
    # Location 0: aPos (vec3)
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)
    # Location 1: aUv (vec2)
    glVertexAttribPointer(1, 2, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(3 * 4))
    glEnableVertexAttribArray(1)
    glBindVertexArray(0)

    # 3. Load texture
    tex_id = load_texture(os.path.join(os.getcwd(), TEXTURE_FILE))

    # 4. Build shaders and query uniform locations
    prog = build_program()
    loc_mv = glGetUniformLocation(prog, "uModelView")
    loc_pj = glGetUniformLocation(prog, "uPersp")
    loc_tex = glGetUniformLocation(prog, "uTexture")
    loc_nlights = glGetUniformLocation(prog, "uNumLights")
    loc_lights = glGetUniformLocation(prog, "uLights")
    num_lights = len(WORLD_LIGHTS)

    # Render loop
    while not glfw.window_should_close(window):
        w, h = glfw.get_framebuffer_size(window)
        glViewport(0, 0, w, h)
        glClearColor(0.6, 0.6, 0.6, 1.0)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        MV = camera.get_view_matrix()
        P = make_perspective(np.radians(40.0), w / max(h, 1), 0.1, 50.0)

        # Transform world-space lights into eye space using current View matrix
        eye_lights = lights_to_eye_space(MV, WORLD_LIGHTS)

        glUseProgram(prog)

        # Upload transformation matrices (transposed for GLSL column-major order)
        glUniformMatrix4fv(loc_mv, 1, GL_FALSE, MV.T.astype(np.float32))
        glUniformMatrix4fv(loc_pj, 1, GL_FALSE, P.T.astype(np.float32))

        # Upload multiple lights
        glUniform1i(loc_nlights, num_lights)
        glUniform3fv(loc_lights, num_lights, eye_lights.flatten())

        # Bind texture to Texture Unit 0
        glActiveTexture(GL_TEXTURE0)
        glBindTexture(GL_TEXTURE_2D, tex_id)
        glUniform1i(loc_tex, 0)

        # Render 3D textured letter
        glBindVertexArray(vao)
        glDrawElements(GL_TRIANGLES, len(indices), GL_UNSIGNED_INT, None)
        glBindVertexArray(0)

        glfw.swap_buffers(window)
        glfw.poll_events()

    # Cleanup
    glDeleteVertexArrays(1, [vao])
    glDeleteBuffers(2, [vbo, ebo])
    glDeleteTextures(1, [tex_id])
    glDeleteProgram(prog)
    glfw.terminate()

In [ ]:
if __name__ == "__main__":
    main()

Loaded RGB texture successfully from: /Users/ncrouch/CPSC5700/HW5/goku.jpg (735x1394)


: 